# P05: Applications and advanced tips

Functions let you name and reuse an analysis. We will write a real one for computing
signal-detection *d'* and use it to cover two common issues: a formula that returns `inf` at the edges, and Python's mutable-default-argument bug.

## A function for d-prime (sensitivity)

In [3]:
from scipy.stats import norm

def dprime(hits, misses, false_alarms, correct_rejections):
    """Signal-detection d': sensitivity in z-units."""
    hit_rate = hits / (hits + misses)
    fa_rate = false_alarms / (false_alarms + correct_rejections)

    # ppf is the percent point function, or the inverse of the cumulative distribution function (CDF)
    return norm.ppf(hit_rate) - norm.ppf(fa_rate)

print(round(dprime(45, 5, 10, 40), 3))

2.123


## What happens at the edges?

In [5]:
# a 'perfect' subject: hit_rate = 1.0  ->  norm.ppf(1.0) = +infinity
print(dprime(50, 0, 10, 40))   # inf, which will run without an error raised unless you explicitly catch it

inf


In [6]:
def dprime_corrected(hits, misses, false_alarms, correct_rejections):
    # correction Hautus, 1995: add 0.5 to each cell and 1 to each total
    hit_rate = (hits + 0.5) / (hits + misses + 1)
    fa_rate = (false_alarms + 0.5) / (false_alarms + correct_rejections + 1)
    return norm.ppf(hit_rate) - norm.ppf(fa_rate)

print(round(dprime_corrected(50, 0, 10, 40), 3))   # a finite, usable value

3.155


:::{admonition} Advanced Tips: the naive d' formula is 'functional but incorrect'
:class: tip
The first `dprime` is exactly the kind of code that you might write if you matched
the textbook formula...and it runs without error. But any subject with a perfect hit
rate (or zero false alarms) yields `inf`, which then messes up every average,
*t*-test, or plot downstream. Whether to apply a correction (and which one) 
is a **scientific** decision a that you must make (and that you need to be aware of if using AI). 
:::

## The mutable default argument bug

In [8]:
# a default list is created once, when the function is defined,
# and then shared across every call that doesn't pass its own list
def add_trial(rt, collected=[]):          # <-- the potential bug is right here
    collected.append(rt)
    return collected

print(add_trial(500))   # [500]
print(add_trial(600))   # [500, 600]: the last call's data came back!

[500]
[500, 600]


In [10]:
# the fix: use None as the default and create a fresh list inside if a list isn't 
# passed to the function to overwrite None
def add_trial(rt, collected=None):
    if collected is None:
        collected = []
    collected.append(rt)
    return collected

print(add_trial(500))   # [500]
print(add_trial(600))   # [600]: independent, as expected

[500]
[600]


:::{admonition} Advanced tip: mutable defaults silently accumulate state
:class: tip
`def f(x, cache=[])` and `def f(x, opts={})` look harmless and run perfectly in a
quick test. But the default object persists between calls, so results from one
participant can leak into the next. Always use `None` as a gaurd and build the
container you want (list/dict) inside the function.
:::

:::{admonition} Advanced tip
:class: tip
Modern function signatures often add **type hints** and **keyword-only arguments**
(everything after a bare `*` must be passed by name), which makes calls 
and the output of the function self-documenting:

```python
def dprime(hits: int, misses: int, *, correction: bool = True) -> float:
    ...
```
:::